In [1]:
##### IMPORT NECESSARY LIBRARIES ##############

import pandas as pd
import sqlite3
import requests
import datetime
import os

#Main library for file handling : pathlib
from pathlib import Path 

In [3]:
###### File Path Setup ##############
DATA = Path("../data") 
data_path = DATA / "DataForAssessment.csv"
df = pd.read_csv(data_path)

In [5]:
###### CHECK DATAFRAME INFO & INCONSISTENCIES ################
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   PacName               138 non-null    object 
 1   RegionName            136 non-null    object 
 2   FieldName             138 non-null    object 
 3   WellName              138 non-null    object 
 4   WellType              138 non-null    object 
 5   RigName               138 non-null    object 
 6   RigType               136 non-null    object 
 7   WaterDepth            138 non-null    float64
 8   Year                  138 non-null    int64  
 9   ReportType            138 non-null    object 
 10  DocumentName          4 non-null      object 
 11  DocumentDate          83 non-null     object 
 12  SubmittedAt           138 non-null    object 
 13  SubmittedBy           138 non-null    object 
 14  AfeCost               138 non-null    float64
 15  AfeDays               1

PacName                   0
RegionName                2
FieldName                 0
WellName                  0
WellType                  0
RigName                   0
RigType                   2
WaterDepth                0
Year                      0
ReportType                0
DocumentName            134
DocumentDate             55
SubmittedAt               0
SubmittedBy               0
AfeCost                   0
AfeDays                   0
SpudDate                  0
WellStartDateTime         0
WellEndDateTime           0
FinalCost                 0
FinalDays                 0
WellNptPercentageWow      0
WellNptPercentage         0
CompletionCostPlan      138
CompletionCostActual    138
DrillingPlanWcpf        138
DrillingActualWcpf      138
dtype: int64

In [7]:
######### CHANGE DATA TYPE OF OBJECT TO DATETIME AND LOCALIZE THE TIMEZONE###############

df['WellStartDateTime'] = pd.to_datetime(df['WellStartDateTime'], utc=True).dt.tz_localize(None)
df['WellEndDateTime'] = pd.to_datetime(df['WellEndDateTime'], utc=True).dt.tz_localize(None)
df['DocumentDate'] = pd.to_datetime(df['DocumentDate'], utc=True).dt.tz_localize(None)
df['SubmittedAt'] = pd.to_datetime(df['WellEndDateTime'], utc=True).dt.tz_localize(None)
df['SpudDate'] = pd.to_datetime(df['SpudDate'], utc=True).dt.tz_localize(None)

df

,PacName,RegionName,FieldName,WellName,WellType,RigName,RigType,WaterDepth,Year,ReportType,DocumentName,DocumentDate,SubmittedAt,SubmittedBy,AfeCost,AfeDays,SpudDate,WellStartDateTime,WellEndDateTime,FinalCost,FinalDays,WellNptPercentageWow,WellNptPercentage,CompletionCostPlan,CompletionCostActual,DrillingPlanWcpf,DrillingActualWcpf
0,PCSB,PM,ANGSI,ANGSI-E16,APPRAISAL CUM DEV/DEVELOPMENT,NAGA-5,JACK UP,69.000000,2019,NOOP,NaN,2019-02-04,2019-02-28 03:00:00,system,8473824.90,24.10,2019-02-10 00:00:00,2019-02-09 13:00:00,2019-02-28 03:00:00,6.283904e+06,18.580000,0.000000,0.000000,NaN,NaN,NaN,NaN
1,PCSB,PM,ANGSI,ANGSI-E16,APPRAISAL CUM DEV/DEVELOPMENT,NAGA-5,JACK UP,69.000000,2019,FWR,NaN,2019-03-15,2019-02-28 03:00:00,system,8473824.90,24.10,2019-02-10 00:00:00,2019-02-09 13:00:00,2019-02-28 03:00:00,6.283904e+06,18.580000,0.000000,0.000000,NaN,NaN,NaN,NaN
2,PCSB,PM,ANGSI,ANGSI-E16,APPRAISAL CUM DEV/DEVELOPMENT,NAGA-5,JACK UP,69.000000,2019,NOOP,NaN,2019-02-04,2019-02-28 03:00:00,system,8473824.90,24.10,2019-02-10 00:00:00,2019-02-09 13:00:00,2019-02-28 03:00:00,6.283904e+06,18.580000,0.000000,0.000000,NaN,NaN,NaN,NaN
3,PCSB,PM,ANGSI,ANGSI-E16,APPRAISAL CUM DEV/DEVELOPMENT,NAGA-5,JACK UP,69.000000,2019,FWR,NaN,2019-03-15,2019-02-28 03:00:00,system,8473824.90,24.10,2019-02-10 00:00:00,2019-02-09 13:00:00,2019-02-28 03:00:00,6.283904e+06,18.580000,0.000000,0.000000,NaN,NaN,NaN,NaN
4,PCSB,PM,ANGSI,ANGSI-E14,APPRAISAL CUM DEV/DEVELOPMENT,NAGA-5,JACK UP,69.000000,2019,NOOP,NaN,2018-10-15,2019-01-05 05:00:00,system,7880044.34,23.10,2018-12-15 00:00:00,2018-12-14 11:30:00,2019-01-05 05:00:00,6.513803e+06,21.730000,11.560000,3.310000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,PCSB,SK,WEST LUTONG A DEV,WEST LUTONG-A08ST3,APPRAISAL CUM DEV/DEVELOPMENT,T-10,PLATFORM TENDER-ASSISTED BARGE TYPE,30.480000,2020,FWR,NaN,NaT,2020-06-24 10:30:00,aem,13892863.88,62.84,2020-02-09 00:00:00,2020-01-29 21:45:00,2020-06-24 10:30:00,1.455986e+07,70.468750,8.001177,7.538803,NaN,NaN,NaN,NaN
134,PTTEP,PM,WPPB INFILL DEV,WPPB-15/15ST1,APPRAISAL CUM DEV/DEVELOPMENT,GUNNLOD,JACK UP,40.200072,2022,NOOP,NaN,NaT,2022-10-10 00:00:00,aem,7999172.03,17.44,2022-06-17 00:00:00,2022-06-17 00:00:00,2022-10-10 00:00:00,1.602606e+07,37.208332,1.259798,1.259798,NaN,NaN,NaN,NaN
135,PTTEP,PM,WPPB INFILL DEV,WPPB-15/15ST1,APPRAISAL CUM DEV/DEVELOPMENT,GUNNLOD,JACK UP,40.200072,2022,FWR,NaN,NaT,2022-10-10 00:00:00,aem,7999172.03,17.44,2022-06-17 00:00:00,2022-06-17 00:00:00,2022-10-10 00:00:00,1.602606e+07,37.208332,1.259798,1.259798,NaN,NaN,NaN,NaN
136,PTTEP,PM,WPPB INFILL DEV,WPPB-14,APPRAISAL CUM DEV/DEVELOPMENT,GUNNLOD,JACK UP,40.200072,2022,NOOP,NaN,NaT,2022-10-31 00:00:00,aem,5866029.56,8.48,2022-06-22 07:15:00,2022-06-21 10:00:00,2022-10-31 00:00:00,9.160501e+06,24.291666,34.905660,34.905660,NaN,NaN,NaN,NaN


In [9]:
############ CHANGE ALL NULL OR EMPTY VALUES TO NA ################
df = df.replace(r'^\s*$', pd.NA, regex=True)
df = df.convert_dtypes()
df

,PacName,RegionName,FieldName,WellName,WellType,RigName,RigType,WaterDepth,Year,ReportType,DocumentName,DocumentDate,SubmittedAt,SubmittedBy,AfeCost,AfeDays,SpudDate,WellStartDateTime,WellEndDateTime,FinalCost,FinalDays,WellNptPercentageWow,WellNptPercentage,CompletionCostPlan,CompletionCostActual,DrillingPlanWcpf,DrillingActualWcpf
0,PCSB,PM,ANGSI,ANGSI-E16,APPRAISAL CUM DEV/DEVELOPMENT,NAGA-5,JACK UP,69.0,2019,NOOP,<NA>,2019-02-04,2019-02-28 03:00:00,system,8473824.9,24.1,2019-02-10 00:00:00,2019-02-09 13:00:00,2019-02-28 03:00:00,6283904.03,18.58,0.0,0.0,<NA>,<NA>,<NA>,<NA>
1,PCSB,PM,ANGSI,ANGSI-E16,APPRAISAL CUM DEV/DEVELOPMENT,NAGA-5,JACK UP,69.0,2019,FWR,<NA>,2019-03-15,2019-02-28 03:00:00,system,8473824.9,24.1,2019-02-10 00:00:00,2019-02-09 13:00:00,2019-02-28 03:00:00,6283904.03,18.58,0.0,0.0,<NA>,<NA>,<NA>,<NA>
2,PCSB,PM,ANGSI,ANGSI-E16,APPRAISAL CUM DEV/DEVELOPMENT,NAGA-5,JACK UP,69.0,2019,NOOP,<NA>,2019-02-04,2019-02-28 03:00:00,system,8473824.9,24.1,2019-02-10 00:00:00,2019-02-09 13:00:00,2019-02-28 03:00:00,6283904.03,18.58,0.0,0.0,<NA>,<NA>,<NA>,<NA>
3,PCSB,PM,ANGSI,ANGSI-E16,APPRAISAL CUM DEV/DEVELOPMENT,NAGA-5,JACK UP,69.0,2019,FWR,<NA>,2019-03-15,2019-02-28 03:00:00,system,8473824.9,24.1,2019-02-10 00:00:00,2019-02-09 13:00:00,2019-02-28 03:00:00,6283904.03,18.58,0.0,0.0,<NA>,<NA>,<NA>,<NA>
4,PCSB,PM,ANGSI,ANGSI-E14,APPRAISAL CUM DEV/DEVELOPMENT,NAGA-5,JACK UP,69.0,2019,NOOP,<NA>,2018-10-15,2019-01-05 05:00:00,system,7880044.34,23.1,2018-12-15 00:00:00,2018-12-14 11:30:00,2019-01-05 05:00:00,6513802.82,21.73,11.56,3.31,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,PCSB,SK,WEST LUTONG A DEV,WEST LUTONG-A08ST3,APPRAISAL CUM DEV/DEVELOPMENT,T-10,PLATFORM TENDER-ASSISTED BARGE TYPE,30.48,2020,FWR,<NA>,NaT,2020-06-24 10:30:00,aem,13892863.88,62.84,2020-02-09 00:00:00,2020-01-29 21:45:00,2020-06-24 10:30:00,14559861.466667,70.46875,8.001177,7.538803,<NA>,<NA>,<NA>,<NA>
134,PTTEP,PM,WPPB INFILL DEV,WPPB-15/15ST1,APPRAISAL CUM DEV/DEVELOPMENT,GUNNLOD,JACK UP,40.200072,2022,NOOP,<NA>,NaT,2022-10-10 00:00:00,aem,7999172.03,17.44,2022-06-17 00:00:00,2022-06-17 00:00:00,2022-10-10 00:00:00,16026063.62,37.208332,1.259798,1.259798,<NA>,<NA>,<NA>,<NA>
135,PTTEP,PM,WPPB INFILL DEV,WPPB-15/15ST1,APPRAISAL CUM DEV/DEVELOPMENT,GUNNLOD,JACK UP,40.200072,2022,FWR,<NA>,NaT,2022-10-10 00:00:00,aem,7999172.03,17.44,2022-06-17 00:00:00,2022-06-17 00:00:00,2022-10-10 00:00:00,16026063.62,37.208332,1.259798,1.259798,<NA>,<NA>,<NA>,<NA>
136,PTTEP,PM,WPPB INFILL DEV,WPPB-14,APPRAISAL CUM DEV/DEVELOPMENT,GUNNLOD,JACK UP,40.200072,2022,NOOP,<NA>,NaT,2022-10-31 00:00:00,aem,5866029.56,8.48,2022-06-22 07:15:00,2022-06-21 10:00:00,2022-10-31 00:00:00,9160500.84,24.291666,34.90566,34.90566,<NA>,<NA>,<NA>,<NA>


In [11]:
########## RENAME THE COLUMNS BASED ON THE TABLE SCHEMA FOR CORRECT INSERTION MAPPING ###############

df = df.rename(columns={

    # PAC / STRUCTURE
    "PacName": "pac_name",
    "RegionName": "region_name",
    "FieldName": "field_name",
    "WellName": "well_name",
    "WellType": "well_type",
    "RigName": "rig_name",
    "RigType": "rig_type",

    # REPORTS
    "ReportType": "report_type",
    "DocumentName": "document_name",
    "DocumentDate": "document_date",
    "SubmittedAt": "submitted_at",
    "SubmittedBy": "submitted_by",
    "Year": "year",

    # OPERATIONS
    "AfeCost": "afe_cost",
    "AfeDays": "afe_days",
    "FinalCost": "final_cost",
    "FinalDays": "final_days",
    "WellNptPercentage": "well_npt_percentage",
    "WellNptPercentageWow": "well_npt_percentage_wow",

    # DRILLING
    "DrillingPlanWcpf": "drilling_plan_wcpf",
    "DrillingActualWcpf": "drilling_actual_wcpf",

    # COMPLETION
    "CompletionCostPlan": "completion_cost_plan",
    "CompletionCostActual": "completion_cost_actual",

    # DATES
    "SpudDate": "spud_date",
    "WellStartDateTime": "well_start_date",
    "WellEndDateTime": "well_end_date"
})

In [13]:
########## REMOVE WHITESPACES OR TRAIL###########
df['pac_name'] = df['pac_name'].str.strip().str.upper()
df['field_name'] = df['field_name'].str.strip().str.upper()
df['region_name'] = df['region_name'].str.strip().str.upper()

In [15]:
# -------------------------------------------------
# REGIONS (NOT NULL: region_name)
# -------------------------------------------------
regions = (
    df[['region_name']]
    .dropna(subset=['region_name'])
    .drop_duplicates()
    .reset_index(drop=True)
)

regions['region_id'] = regions.index + 1
regions = regions[['region_id', 'region_name']]

# -------------------------------------------------
# PAC (NOT NULL: pac_name, region_name)
# -------------------------------------------------
pac = (
    df[['pac_name', 'region_name']]
    .dropna(subset=['pac_name', 'region_name'])
    .drop_duplicates()
)

pac = pac.merge(regions, on='region_name', how='left')

pac = pac.reset_index(drop=True)
pac['pac_id'] = pac.index + 1
pac = pac[['pac_id', 'pac_name', 'region_id']]

# -------------------------------------------------
# FIELDS (NOT NULL: field_name, pac_name)
# -------------------------------------------------
fields = (
    df[['field_name', 'pac_name']]
    .dropna(subset=['field_name', 'pac_name'])
    .drop_duplicates()
)

fields = fields.merge(
    pac[['pac_id', 'pac_name']],
    on='pac_name',
    how='left'
)

fields = fields.reset_index(drop=True)
fields['field_id'] = fields.index + 1
fields = fields[['field_id', 'field_name', 'pac_id']]

# -------------------------------------------------
# WELLS (NOT NULL: well_name, field_name)
# -------------------------------------------------
wells = (
    df[['well_name', 'well_type', 'well_start_date', 'well_end_date', 'field_name']]
    .dropna(subset=['well_name', 'field_name'])
    .drop_duplicates()
)

wells = wells.merge(
    fields[['field_id', 'field_name']],
    on='field_name',
    how='left'
)

wells = wells.reset_index(drop=True)
wells['well_id'] = wells.index + 1
wells = wells[['well_id', 'well_name', 'well_type', 'well_start_date', 'well_end_date','field_id']]

# -------------------------------------------------
# RIGS (NOT NULL: rig_name)
# -------------------------------------------------
rigs = (
    df[['rig_name', 'rig_type']]
    .dropna(subset=['rig_name'])
    .drop_duplicates()
    .reset_index(drop=True)
)

rigs['rig_id'] = rigs.index + 1
rigs = rigs[['rig_id', 'rig_name', 'rig_type']]

# -------------------------------------------------
# WELL-RIG ASSIGNMENT (NOT NULL: well_name, rig_name)
# -------------------------------------------------
well_rig_assignment = (
    df[['well_name', 'rig_name']]
    .dropna(subset=['well_name', 'rig_name'])
    .drop_duplicates()
)

well_rig_assignment = well_rig_assignment.merge(wells[['well_id', 'well_name']],on='well_name',how='left')
well_rig_assignment = well_rig_assignment.merge(rigs[['rig_id', 'rig_name']],on='rig_name',how='left')

well_rig_assignment = well_rig_assignment.reset_index(drop=True)
well_rig_assignment['assignment_id'] = well_rig_assignment.index + 1

well_rig_assignment['start_date'] = pd.NaT
well_rig_assignment['end_date'] = pd.NaT

well_rig_assignment = well_rig_assignment[
    ['assignment_id', 'well_id', 'rig_id', 'start_date', 'end_date']
]

# -------------------------------------------------
# REPORTS (NOT NULL: well_name, report_type, document_date)
# -------------------------------------------------
reports = (
    df[[
        'well_name',
        'report_type',
        'document_name',
        'document_date',
        'year',
        'submitted_at',
        'submitted_by'
    ]]
    .dropna(subset=['well_name', 'report_type', 'document_date'])
    .drop_duplicates()
)

reports = reports.merge(wells[['well_id', 'well_name']],on='well_name',how='left')

reports = reports.reset_index(drop=True)
reports['report_id'] = reports.index + 1

reports = reports[
    [
        'report_id',
        'well_id',
        'report_type',
        'document_name',
        'document_date',
        'year',
        'submitted_at',
        'submitted_by'
    ]
]

# -------------------------------------------------
# WELL OPERATIONS (NOT NULL: well_name)
# -------------------------------------------------
well_operations = (
    df[[
        'well_name',
        'afe_cost',
        'afe_days',
        'final_cost',
        'final_days',
        'well_npt_percentage',
        'well_npt_percentage_wow'
    ]]
    .dropna(subset=['well_name'])
    .drop_duplicates()
)

well_operations = well_operations.merge(wells[['well_id', 'well_name']],on='well_name',how='left')

well_operations = well_operations.reset_index(drop=True)
well_operations['operation_id'] = well_operations.index + 1

# -------------------------------------------------
# DRILLING (NOT NULL: well_name)
# -------------------------------------------------
drilling = (
    df[[
        'well_name', 'spud_date',
        'drilling_plan_wcpf',
        'drilling_actual_wcpf'
    ]]
    .dropna(subset=['well_name'])
    .drop_duplicates()
)

drilling = drilling.merge(
    well_operations[['operation_id', 'well_name']],
    on='well_name',
    how='left'
)

drilling = drilling.reset_index(drop=True)
drilling['drilling_id'] = drilling.index + 1

drilling = drilling[
    [
        'drilling_id',
        'operation_id', 'spud_date',
        'drilling_plan_wcpf',
        'drilling_actual_wcpf'
    ]
]

# -------------------------------------------------
# COMPLETION (NOT NULL: well_name)
# -------------------------------------------------
completion = (
    df[[
        'well_name',
        'completion_cost_plan',
        'completion_cost_actual'
    ]]
    .dropna(subset=['well_name'])
    .drop_duplicates()
)

completion = completion.merge(
    well_operations[['operation_id', 'well_name']],
    on='well_name',
    how='left'
)

completion = completion.reset_index(drop=True)
completion['completion_id'] = completion.index + 1

completion = completion[
    [
        'completion_id',
        'operation_id',
        'completion_cost_plan',
        'completion_cost_actual'
    ]
]

In [19]:
############ DEFINE NULL DEPENDENCIES ##############

#********** Null region name is still kept and marked as unknown as it tied to other fact table
UNKNOWN_REGION_ID = 999

df['region_name'] = df['region_name'].fillna('UNKNOWN')

if 'region_name' in regions.columns:
    if not (regions['region_name'] == 'UNKNOWN').any():
        regions = pd.concat([
            regions,
            pd.DataFrame({'region_id': [UNKNOWN_REGION_ID], 'region_name': ['UNKNOWN']})
        ], ignore_index=True)

In [27]:
############ REMOVE UNNECESSARY COLUMN FROM DF BEFORE INSERT INTO DB ###############

well_operations = well_operations.drop(columns=['well_name'])
well_operations = well_operations.drop(columns=['well_name'])

In [25]:
########## INSERT DATA INTO SQLITE TABLES #####################

import logging

#Logging setup
logging.basicConfig(level=logging.INFO,format="%(asctime)s - %(levelname)s - %(message)s")

DB_NAME = "drilling_operations.db"


def load_table(conn, df, table_name):
    try:
        logging.info(f"Loading table: {table_name}")

        df.to_sql(
            table_name,
            conn,
            if_exists="append",
            index=False
        )

        logging.info(f"SUCCESS: {table_name} loaded ({len(df)} rows)")

    except Exception as e:
        logging.error(f"FAILED: {table_name} -> {str(e)}")
        raise


#Validate insertion
def validate_table(conn, table_name):
    try:
        count = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {table_name}", conn)
        logging.info(f"VALIDATION: {table_name} has {count['cnt'][0]} rows")

    except Exception as e:
        logging.error(f"Validation failed for {table_name}: {str(e)}")


# Main Load process
conn = sqlite3.connect(DB_NAME)
conn.execute("PRAGMA foreign_keys = ON;")

try:
    logging.info("===== STARTING DATA LOAD =====")

    load_table(conn, regions, "regions")
    load_table(conn, pac, "pac")
    load_table(conn, fields, "fields")
    load_table(conn, wells, "wells")
    load_table(conn, rigs, "rigs")
    load_table(conn, well_rig_assignment, "well_rig_assignment")
    load_table(conn, well_operations, "well_operations")
    load_table(conn, drilling, "drilling")
    load_table(conn, completion, "completion")
    load_table(conn, reports, "reports")

    logging.info("===== DATA LOAD COMPLETE =====")


    logging.info("===== RUNNING VALIDATION =====")

    tables = [
        "regions",
        "pac",
        "fields",
        "wells",
        "rigs",
        "well_rig_assignment",
        "well_operations",
        "drilling",
        "completion",
        "reports"
    ]

    for t in tables:
        validate_table(conn, t)

    logging.info("===== VALIDATION COMPLETE =====")

except Exception as e:
    logging.error(f"PIPELINE FAILED: {str(e)}")

finally:
    conn.close()
    logging.info("Database connection closed")

2026-06-01 19:43:54,229 - INFO - ===== STARTING DATA LOAD =====
2026-06-01 19:43:54,231 - INFO - Loading table: regions
2026-06-01 19:43:54,246 - INFO - SUCCESS: regions loaded (3 rows)
2026-06-01 19:43:54,248 - INFO - Loading table: pac
2026-06-01 19:43:54,265 - INFO - SUCCESS: pac loaded (18 rows)
2026-06-01 19:43:54,266 - INFO - Loading table: fields
2026-06-01 19:43:54,286 - INFO - SUCCESS: fields loaded (58 rows)
2026-06-01 19:43:54,288 - INFO - Loading table: wells
2026-06-01 19:43:54,307 - INFO - SUCCESS: wells loaded (122 rows)
2026-06-01 19:43:54,309 - INFO - Loading table: rigs
2026-06-01 19:43:54,324 - INFO - SUCCESS: rigs loaded (29 rows)
2026-06-01 19:43:54,326 - INFO - Loading table: well_rig_assignment
2026-06-01 19:43:54,345 - INFO - SUCCESS: well_rig_assignment loaded (125 rows)
2026-06-01 19:43:54,347 - INFO - Loading table: well_operations
2026-06-01 19:43:54,366 - INFO - SUCCESS: well_operations loaded (122 rows)
2026-06-01 19:43:54,368 - INFO - Loading table: drill